In [ ]:
# Task 1

In [ ]:
import numpy as np

a = np.array([0.1, 0.2, 0.3], dtype=np.float32)
b = np.array([0.1, 0.2, 0.3], dtype=np.float64)

sum32 = np.float32(0.0)
sum64 = np.float64(0.0)

for i in a:
  sum32 += i
for j in b:
  sum64 += j

print(sum32, sum64)

In [ ]:
# Task 2

In [ ]:
import numpy as np

A = np.arange(-10, 10).reshape(4, 5)

print(A[1], "-1.1") #1
print(A[:, 3], "-1.2") 
print(A[0:3, 1:4], "-1.3")
print(A[np.ix_([0, 2, 3], [1, 4])], "-2") #2

A[A < 0] = 0 #3
print(A, "-3") 

B = np.arange(4*1).reshape(4, 1) #4
C = np.arange(1*5).reshape(1, 5)

res = B + C

print(res, "-4")

In [ ]:
# Task 3

In [ ]:
import numpy as np

filename = 'memmap_array.dat'
shape = (1000, 1000)

fp = np.memmap(filename, dtype='float64', mode='w+', shape=shape)

rng = np.random.default_rng()
fp[:] = rng.random(shape)

fp.flush()

del fp

In [ ]:
import numpy as np

filename = 'memmap_array.dat'
shape = (1000, 1000)

fp = np.memmap(filename, dtype='float64', mode='r', shape=shape)

row = fp[78, :]

col = fp[:, 78]

print("Форма строки:", row.shape, "Первые 23 элемента:", row[:23])
print("Форма столбца:", col.shape, "Первые 23 элемента:", col[:23])

del fp

In [ ]:
import numpy as np

filename = 'memmap_array.dat'
shape = (1000, 1000)

fp = np.memmap(filename, dtype='float64', mode='r', shape=shape)

def iter_rows(a):
    total = 0.0
    for i in range(a.shape[0]):
        total += a[i, :].sum()
    return total

def iter_cols(a):
    total = 0.0
    for j in range(a.shape[1]):
        total += a[:, j].sum()
    return total

print("Обход файла по строкам:")
%timeit iter_rows(fp)

print("\nОбход файла по столбцам:")
%timeit iter_cols(fp)
print("\n - дольше, так как в памяти массив расположен построчно")

del fp

In [ ]:
# Task 4

In [ ]:
import numpy as np
from scipy.sparse import random

matrix = random(5, 5, density=0.2, format='csr', random_state=13)

print("Тип объекта:", type(matrix))
print("Количество ненулевых элементов:", matrix.nnz)

print("\nВнутреннее представление CSR (координаты и значения):")
print(matrix)

print("\nПлотный вид:")
print(np.round(matrix.toarray(), 2))

coo_matrix = matrix.tocoo()

csc_matrix = matrix.tocsc()

print("Тип:", type(coo_matrix))
print("Строки:", coo_matrix.row)
print("Столбцы:", coo_matrix.col)
print("Значения:", np.round(coo_matrix.data, 2))

print("\nТип:", type(csc_matrix))
print("Индексы строк:", csc_matrix.indices)
print("Указатели столбцов:", csc_matrix.indptr)
print("Значения:", np.round(csc_matrix.data, 2))

vector = np.arange(1.0, 6.0)

res = matrix @ vector

print("\nТип результата:", type(res))
print("Форма результата:", res.shape)
print("Результат умножения:", np.round(res, 3))

In [ ]:
import numpy as np
from scipy.sparse import random

size = 1000
density = 0.01

A_sparse = random(size, size, density=density, format='csr', random_state=42)
A_dense = A_sparse.toarray()

v = np.random.rand(size)

print("Разреженная (CSR @ v):")
%timeit A_sparse @ v

print("Плотная (NumPy @ v):")
%timeit A_dense @ v

In [ ]:
# Task 5

In [ ]:
%pip install -q --upgrade cython setuptools wheel numpy

In [1]:
%load_ext cython

In [7]:
%%writefile sum_sq_cy.pyx
import cython

@cython.boundscheck(False)
@cython.wraparound(False)
def sum_of_squares_cy(double[:] arr):
    cdef Py_ssize_t i
    cdef Py_ssize_t n = arr.shape[0]
    cdef double total = 0.0
    
    for i in range(n):
        total += arr[i] * arr[i]
        
    return total

Overwriting sum_sq_cy.pyx


In [9]:
import sys

!{sys.executable} -m Cython.Build.Cythonize -i -3 sum_sq_cy.pyx

Compiling C:\Users\User\sum_sq_cy.pyx because it changed.
[1/1] Cythonizing C:\Users\User\sum_sq_cy.pyx
sum_sq_cy.c
   ‘®§¤ Ґвбп ЎЁЎ«Ё®вҐЄ  C:\Users\User\tmpl_l89tmd\Release\Users\User\sum_sq_cy.cp314-win_amd64.lib Ё ®ЎкҐЄв C:\Users\User\tmpl_l89tmd\Release\Users\User\sum_sq_cy.cp314-win_amd64.exp
‘®§¤ ­ЁҐ Є®¤ 
‘®§¤ ­ЁҐ Є®¤  § ўҐаиҐ­®


In [11]:
import numpy as np
from sum_sq_cy import sum_of_squares_cy

def sum_of_squares_py(arr):
    total = 0.0
    for x in arr:
        total += x * x
    return total


N = 10**7
np_arr = np.random.random(N)
py_list = np_arr.tolist()

print("Python:")
%timeit sum_of_squares_py(py_list)

print("\nCython:")
%timeit sum_of_squares_cy(np_arr)

Python:
578 ms ± 3.21 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Cython:
10.2 ms ± 22.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
